<h2>O que é o Pandera?</h2>

<p>O Pandera é uma biblioteca Python focada na validação de estruturas e tipos de dados em DataFrames, como Pandas, Polars, PySpark e Dask. Ele garante que os dados que entram ou saem do seu pipeline sigam regras antes de serem usados em análises ou inserções no banco de dados.</p>

<h3>Para que serve?</h3>

<ul>
    <li><strong>Garantia de qualidade dos dados:</strong> verifica se as colunas obrigatórias existem e se os tipos estão corretos.</li>
    <li><strong>Validação de regras de negócio:</strong> checa se os valores atendem aos critérios esperados, como idades positivas e e-mails válidos.</li>
    <li><strong>Documentação viva:</strong> o próprio esquema de validação documenta o formato esperado dos dados.</li>
</ul>


In [1]:
%pip install pandera

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import pandera as pa
from pandera import Column, Check

# Definindo o esquema com coerce (conversão de tipo automática) e regras
schema = pa.DataFrameSchema(
    columns={
        "id": Column(int, unique=True, nullable=False),
        "nome": Column(str, checks=[Check.str_length(min_value=2)]),
        "categoria": Column(str, checks=[Check.isin(["A", "B", "C"])]),
        "preco": Column(float, checks=[Check.greater_than(0.0)]),
        "desconto": Column(float, checks=[Check.in_range(0.0, 1.0)], nullable=True),
        "data_cadastro": Column(pa.DateTime, nullable=False),
    },
    coerce=True,       # Converte strings/tipos compatíveis automaticamente
    strict=True        # Rejeita colunas não mapeadas no esquema
)

# Dados de teste
df = pd.DataFrame({
    "id": [1, 2, 3],
    "nome": ["Mouse", "Teclado", "Monitor"],
    "categoria": ["A", "B", "A"],
    "preco": ["50.0", "150.5", "899.90"], # strings numéricas serão convertidas
    "desconto": [0.1, None, 0.05],
    "data_cadastro": ["2026-01-10", "2026-02-15", "2026-03-01"]
})

df_validado = schema.validate(df)
print("DataFrame validado com sucesso!")

DataFrame validado com sucesso!


c:\Users\Luasa\anaconda3\Lib\site-packages\pandera\_pandas_deprecated.py:144: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [7]:
import pandas as pd
import pandera as pa
from pandera.typing import Series

class VendasModel(pa.DataFrameModel):
    id_venda: int = pa.Field(unique=True, ge=1)
    cliente: str = pa.Field(str_length={"min_value": 3})
    status: str = pa.Field(isin=["PENDENTE", "PAGO", "CANCELADO"])
    valor_total: float = pa.Field(gt=0)
    
    class Config:
        coerce = True
        strict = True

# Validação:
df_vendas = pd.DataFrame({
    "id_venda": [101, 102],
    "cliente": ["Mariana", "Lucas"],
    "status": ["PAGO", "PENDENTE"],
    "valor_total": [250.0, 80.0]
})

df_validado = VendasModel.validate(df_vendas)

In [8]:
def valida_cpf(series: pd.Series) -> pd.Series:
    # Apenas dígitos e comprimento 11
    return series.astype(str).str.replace(r"\D", "", regex=True).str.len() == 11

schema_cliente = pa.DataFrameSchema({
    "cpf": Column(str, checks=Check(valida_cpf, error="CPF deve conter 11 dígitos"))
})

In [9]:
schema_projetos = pa.DataFrameSchema(
    columns={
        "data_inicio": Column(pa.DateTime),
        "data_fim": Column(pa.DateTime)
    },
    checks=[
        Check(
            lambda df: df["data_fim"] >= df["data_inicio"],
            error="A data de fim deve ser posterior ou igual à data de início"
        )
    ]
)

In [10]:
from pandera.typing import DataFrame

@pa.check_types
def calcular_imposto(df: DataFrame[VendasModel]) -> DataFrame[VendasModel]:
    # Faz transformações garantindo a integridade dos dados
    df["valor_total"] = df["valor_total"] * 1.1
    return df

- Em pipelines de dados (Airflow, scripts de ingestão), é essencial registrar todos os registros inválidos sem quebrar a execução imediatamente:

In [11]:
import pandera as pa
import pandas as pd

df_com_problemas = pd.DataFrame({
    "id_venda": [1, 1],                # Erro: duplicado
    "cliente": ["Al", "Bob"],          # Erro: tamanho < 3
    "status": ["INVALIDO", "PAGO"],    # Erro: fora da lista
    "valor_total": [-10.0, 50.0]       # Erro: negativo
})

try:
    VendasModel.validate(df_com_problemas, lazy=True)
except pa.errors.SchemaErrors as exc:
    # DataFrame consolidado com todas as falhas
    relatorio_erros = exc.failure_cases
    print(relatorio_erros[["column", "check", "failure_case", "index"]])

        column                                    check failure_case  index
0     id_venda                         field_uniqueness            1      0
1     id_venda                         field_uniqueness            1      1
2      cliente                      str_length(3, None)           Al      0
3       status  isin(['PENDENTE', 'PAGO', 'CANCELADO'])     INVALIDO      0
4  valor_total                          greater_than(0)        -10.0      0


%md-sandbox
### Comparando Duas Abordagens

Existem duas maneiras complementares de pensar sobre esse problema:

<style>
.flip-card-a, .flip-card-b {
perspective: 1000px; 
flex: 1; 
height: 280px; 
cursor: pointer;
}
.flip-card-a .flip-inner, .flip-card-b .flip-inner {
position: relative; 
width: 100%; 
height: 100%; 
transition: transform 0.6s; 
transform-style: preserve-3d;
}
.flip-card-a:hover .flip-inner { transform: rotateY(180deg); }
.flip-card-b:hover .flip-inner { transform: rotateY(180deg); }
.flip-front, .flip-back {
position: absolute; 
width: 100%; 
height: 100%; 
backface-visibility: hidden; 
border-radius: 8px; 
overflow: hidden;
}
.flip-back { transform: rotateY(180deg); }
</style>

<div style="display: flex; gap: 16px; margin: 16px 0;">

<div class="flip-card-a">
<div class="flip-inner">
<div class="flip-front" style="border: 2px solid #1565C0;">
<div style="background: #1565C0; color: #fff; padding: 12px 16px; font-weight: bold; font-size: 1.05em;">Abordagem A</div>
<div style="padding: 20px; display: flex; flex-direction: column; align-items: center; justify-content: center; height: calc(100% - 46px); background: #fff;">
<div style="font-size: 48px; margin-bottom: 12px;">🏢</div>
<p style="margin: 0; color: #333; text-align: center; font-size: 1em;">Um método abrangente, do tipo *top-down*, que estabelece limites claros no nível mais alto e funciona bem quando as divisões são óbvias e estáveis.</p>
<div style="margin-top: 16px; font-size: 0.85em; color: #999;">Passe o mouse para ver detalhes</div>
</div>
</div>
<div class="flip-back" style="border: 2px solid #1565C0;">
<div style="background: #1565C0; color: #fff; padding: 12px 16px; font-weight: bold; font-size: 1.05em;">Abordagem A</div>
<div style="padding: 16px; background: #fff; height: calc(100% - 46px);">
<ul style="margin: 0; padding-left: 20px; color: #333;">
<li>Simples de implementar, com configuração mínima</li>
<li>Responsabilidade clara em nível de grupo ou divisão</li>
<li>Funciona melhor quando as fronteiras são estáveis ​​e bem definidas</li>
<li>Menor flexibilidade para distinções detalhadas</li>
</ul>
<div style="margin-top: 16px; padding: 8px 12px; background: #E3F2FD; border-radius: 4px; font-size: 0.9em; color: #0D47A1;">
<strong>Ideal para:</strong> Separação de alto nível entre grandes grupos ou unidades
</div>
</div>
</div>
</div>
</div>

<div class="flip-card-b">
<div class="flip-inner">
<div class="flip-front" style="border: 2px solid #2E7D32;">
<div style="background: #2E7D32; color: #fff; padding: 12px 16px; font-weight: bold; font-size: 1.05em;">Abordagem B</div>
<div style="padding: 20px; display: flex; flex-direction: column; align-items: center; justify-content: center; height: calc(100% - 46px); background: #fff;">
<div style="font-size: 48px; margin-bottom: 12px;">🏷️</div>
<p style="margin: 0; color: #333; text-align: center; font-size: 1em;">Um método flexível, do tipo *bottom-up* (de baixo para cima), que associa rótulos ou metadados a itens individuais, permitindo a geração de relatórios em qualquer nível de granularidade.</p>
<div style="margin-top: 16px; font-size: 0.85em; color: #999;">Passe o mouse para ver detalhes</div>
</div>
</div>
<div class="flip-back" style="border: 2px solid #2E7D32;">
<div style="background: #2E7D32; color: #fff; padding: 12px 16px; font-weight: bold; font-size: 1.05em;">Abordagem B</div>
<div style="padding: 16px; background: #fff; height: calc(100% - 46px);">
<ul style="margin: 0; padding-left: 20px; color: #333;">
<li>Atribui identificadores a itens ou recursos individuais</li>
<li>Suporta visões transversais que abrangem múltiplos grupos</li>
<li>Exige mais configuração inicial, mas é muito mais flexível</li>
<li>Pode ser aplicada retroativamente conforme as necessidades evoluem</li>
</ul>
<div style="margin-top: 16px; padding: 8px 12px; background: #E8F5E9; border-radius: 4px; font-size: 0.9em; color: #1B5E20;">
<strong>Ideal para:</strong> Rastreamento granular em qualquer dimensão – projetos, equipes, categorias
</div>
</div>
</div>
</div>
</div>

</div>

<div style="border-left: 4px solid #1976d2; background: #e65100; padding: 12px 16px; border-radius: 4px; margin: 16px 0;">
<div style="color: #0d47a1; font-weight: bold; margin-bottom: 6px;">&#9432; Nota</div>
<div>Estas duas abordagens não são mutuamente exclusivas. O padrão mais comum é utilizar a Abordagem A para divisões principais e a Abordagem B para distinções mais granulares dentro dessas divisões.</div>
</div>